<a href="https://colab.research.google.com/github/slomi23/NLP_final_project/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
import json
import sys
import os
from pathlib import Path
import torch
import numpy as np

# Add the project root to the path so we can import from src/
# Assuming this notebook is in notebooks/ and src/ is in the root
project_root = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(project_root))

# Import the encoder components
from src.models.encoder import EncoderOnlyTransformer, SimpleTokenizer, train_encoder, contrastive_loss

# 1. Define paths
CHUNKS_FILE = project_root / "data" / "jurafsky_chunks" / "chunks.jsonl"
MODEL_SAVE_PATH = project_root / "models" / "encoder_transformer.pth"

# Ensure models directory exists
MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"📂 Project Root: {project_root}")
print(f"📄 Chunks File: {CHUNKS_FILE}")

# 2. Load chunks from JSONL
def load_chunks(filepath):
    chunks = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                chunks.append(json.loads(line.strip()))
    return chunks

chunks = load_chunks(CHUNKS_FILE)
print(f"✅ Loaded {len(chunks)} chunks")

# 3. Prepare training data
# The train_encoder function expects a list of dicts with 'passage_text' key
training_data = [{'passage_text': chunk['text']} for chunk in chunks]
print(f"✅ Prepared {len(training_data)} training samples")

# Quick sanity check on data length
avg_len = np.mean([len(d['passage_text'].split()) for d in training_data])
print(f"📏 Average word count per chunk: {avg_len:.1f}")


📂 Project Root: c:\Users\salo\NLP_final_project
📄 Chunks File: c:\Users\salo\NLP_final_project\data\jurafsky_chunks\chunks.jsonl
✅ Loaded 1730 chunks
✅ Prepared 1730 training samples
📏 Average word count per chunk: 202.7


In [78]:
import pandas as pd

# 1. Load ArXiv Data
ARXIV_FILE = project_root / "data" / "processed" / "arxiv_cs_papers_processed.csv"

print(f"📚 Loading ArXiv papers from {ARXIV_FILE}...")
arxiv_df = pd.read_csv(ARXIV_FILE)

# 2. Clean and Prepare ArXiv Data
# We need 'title' and 'abstract'. Let's combine them into a single text block.
# This helps the model learn the connection between titles and their content.

def clean_and_combine(row):
    # Handle missing values
    title = str(row['title']) if pd.notna(row['title']) else ""
    abstract = str(row['abstract']) if pd.notna(row['abstract']) else ""
    
    # Combine: "Title: ... Abstract: ..."
    combined_text = f"Title: {title}. Abstract: {abstract}"
    
    # Basic cleaning: remove excessive whitespace
    import re
    combined_text = re.sub(r'\s+', ' ', combined_text).strip()
    
    return combined_text if len(combined_text) > 10 else None

arxiv_df['combined_text'] = arxiv_df.apply(clean_and_combine, axis=1)

# Drop rows where cleaning resulted in empty text
arxiv_df = arxiv_df.dropna(subset=['combined_text'])

print(f"✅ Loaded {len(arxiv_df)} ArXiv papers")

# 3. Convert to training format
# Format: {'passage_text': '...'}
arxiv_training_data = [
    {'passage_text': text} 
    for text in arxiv_df['combined_text'].tolist()
]

print(f"✅ Prepared {len(arxiv_training_data)} ArXiv samples")

# 4. Combine with Jurafsky Chunks
# Now we have two sources of high-quality NLP/CS text
all_training_data = training_data + arxiv_training_data

print(f"\n📊 Total Training Dataset:")
print(f"   - Jurafsky Chunks: {len(training_data)}")
print(f"   - ArXiv Papers:    {len(arxiv_training_data)}")
print(f"   - Total:           {len(all_training_data)}")

# Quick sanity check on combined data length
avg_len_combined = np.mean([len(d['passage_text'].split()) for d in all_training_data])
print(f"📏 Average word count per sample: {avg_len_combined:.1f}")


📚 Loading ArXiv papers from c:\Users\salo\NLP_final_project\data\processed\arxiv_cs_papers_processed.csv...
✅ Loaded 417331 ArXiv papers
✅ Prepared 417331 ArXiv samples

📊 Total Training Dataset:
   - Jurafsky Chunks: 1730
   - ArXiv Papers:    417331
   - Total:           419061
📏 Average word count per sample: 188.7


In [79]:
import subprocess
import json
from pathlib import Path
import sys

# 1. Run the pair creation script
# This generates data/processed/train_pairs.jsonl, val_pairs.jsonl, test_pairs.jsonl
print("🔄 Running make_pairs.py to generate training pairs...")
try:
    # Run the script from the project root context
    result = subprocess.run(
        [sys.executable, "src/data/make_pairs.py"],
        cwd=project_root,
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"❌ Error running make_pairs.py: {e.stderr}")
    raise

# 2. Load the generated training pairs
TRAIN_PAIRS_FILE = project_root / "data" / "processed" / "train_pairs.jsonl"

def load_pairs(filepath):
    pairs = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                pairs.append(json.loads(line.strip()))
    return pairs

print(f"📚 Loading training pairs from {TRAIN_PAIRS_FILE}...")
train_pairs = load_pairs(TRAIN_PAIRS_FILE)
print(f"✅ Loaded {len(train_pairs)} training pairs")

# 3. Convert pairs to the format expected by train_encoder
# train_encoder expects: [{'passage_text': '...'}, ...]
# However, for contrastive learning, we usually need both query and passage.
# Looking at your encoder.py's train_encoder:
# It only uses 'passage_text' from the dict.
# It creates "positive pairs" by shuffling embeddings internally.
# This is a valid self-supervised approach (like SimCLR).

# So, we just need to extract the 'positive' text from our pairs
# and format it as {'passage_text': text}

training_data_from_pairs = [
    {'passage_text': pair['positive']} 
    for pair in train_pairs
]

print(f"✅ Prepared {len(training_data_from_pairs)} samples from pairs")

# 4. Combine ALL sources for maximum robustness
# 1. Jurafsky Chunks (domain specific, long text)
# 2. ArXiv Papers (domain specific, dense info)
# 3. MS MARCO Pairs (query-relevance focus)

all_training_data = training_data + arxiv_training_data + training_data_from_pairs

print(f"\n📊 Final Combined Training Dataset:")
print(f"   - Jurafsky Chunks: {len(training_data)}")
print(f"   - ArXiv Papers:    {len(arxiv_training_data)}")
print(f"   - MS MARCO Pairs:  {len(training_data_from_pairs)}")
print(f"   - TOTAL:           {len(all_training_data)}")

# Sanity check
if len(all_training_data) == 0:
    raise ValueError("No training data found! Check your data files.")
else:
    print("✅ Data ready for training!")


🔄 Running make_pairs.py to generate training pairs...
Loading data вЂ¦
Building pairs вЂ¦
  total pairs: 20,000
  train/val/test: 16000/2000/2000
  wrote 16,000 pairs в†’ data\processed\train_pairs.jsonl
  wrote 2,000 pairs в†’ data\processed\val_pairs.jsonl
  wrote 2,000 pairs в†’ data\processed\test_pairs.jsonl
Done вњ“

📚 Loading training pairs from c:\Users\salo\NLP_final_project\data\processed\train_pairs.jsonl...
✅ Loaded 16000 training pairs
✅ Prepared 16000 samples from pairs

📊 Final Combined Training Dataset:
   - Jurafsky Chunks: 1730
   - ArXiv Papers:    417331
   - MS MARCO Pairs:  16000
   - TOTAL:           435061
✅ Data ready for training!


In [80]:
import json
import random
import re
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load MS MARCO Data Directly
print("📚 Loading MS MARCO data directly from JSON files...")

QUERIES_FILE = project_root / "data" / "processed" / "msmarco_small_queries.json"
PASSAGES_FILE = project_root / "data" / "processed" / "msmarco_small_passages.json"
QRELS_FILE = project_root / "data" / "processed" / "msmarco_small_qrels.json"

with open(QUERIES_FILE, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

with open(PASSAGES_FILE, 'r', encoding='utf-8') as f:
    passages_list = json.load(f)

with open(QRELS_FILE, 'r', encoding='utf-8') as f:
    qrels = json.load(f)

print(f"✅ Loaded {len(queries_list)} queries")
print(f"✅ Loaded {len(passages_list)} passages")
print(f"✅ Loaded {len(qrels)} relevance judgments")

# 2. Create Lookups
query_lookup = {q['id']: q['text'] for q in queries_list}
passage_lookup = {p['id']: p['text'] for p in passages_list}

# 3. Generate MS MARCO Triplets
print("🔄 Generating MS MARCO triplets...")
ms_marco_triplets = []

for query_id, relevant_passages in qrels.items():
    if query_id not in query_lookup:
        continue
        
    query_text = query_lookup[query_id]
    
    # Get the best relevant passage (highest score)
    best_pid, _ = max(relevant_passages, key=lambda x: x)
    
    if best_pid not in passage_lookup:
        continue
        
    positive_text = passage_lookup[best_pid]
    
    # Filter short queries
    if len(query_text.split()) < 3:
        continue
        
    ms_marco_triplets.append({
        'query': query_text,
        'positive': positive_text,
        'negatives': [] # Will add negatives later
    })

print(f"✅ Generated {len(ms_marco_triplets)} MS MARCO pairs")

# 4. Load ArXiv Triplets
print("📚 Loading ArXiv triplets...")
ARXIV_TRAIN_FILE = project_root / "data" / "processed" / "arxiv_train_triplets.jsonl"
arxiv_triplets = []
with open(ARXIV_TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            arxiv_triplets.append(json.loads(line.strip()))

print(f"✅ Loaded {len(arxiv_triplets)} ArXiv triplets")

# 5. Generate Jurafsky Book Triplets (Synthetic)
print("📚 Generating Jurafsky Book triplets...")

# Load Jurafsky chunks
chunks_file = project_root / "data" / "jurafsky_chunks" / "chunks.jsonl"
chunks = []
with open(chunks_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line.strip()))

# Helper to extract sentences
def extract_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    valid_sentences = [
        s.strip() for s in sentences 
        if len(s.split()) > 5 and not s.startswith('(') and not s.startswith('[')
    ]
    return valid_sentences

jurafsky_triplets = []
for chunk in chunks:
    text = chunk['text']
    sentences = extract_sentences(text)
    
    for sentence in sentences:
        if len(sentence.split()) >= 10:
            jurafsky_triplets.append({
                'query': sentence,
                'positive': text,
                'negatives': [] # Will add negatives later
            })

print(f"✅ Generated {len(jurafsky_triplets)} Jurafsky triplets")

# 6. Prepare Negative Pool
print("🔄 Preparing negative pool...")

# Load ArXiv data for negatives
arxiv_file = project_root / "data" / "processed" / "arxiv_cs_papers_processed.csv"
arxiv_df = pd.read_csv(arxiv_file)
arxiv_df['combined_text'] = arxiv_df.apply(
    lambda row: f"Title: {row['title']}. Abstract: {row['abstract']}" if pd.notna(row['title']) and pd.notna(row['abstract']) else "",
    axis=1
)

# Pool includes ArXiv passages + Jurafsky chunks
arxiv_pool = arxiv_df['combined_text'].tolist()
jurafsky_pool = [chunk['text'] for chunk in chunks]
negative_pool = arxiv_pool + jurafsky_pool

print(f"✅ Negative pool size: {len(negative_pool)} passages")

# 7. Add Negatives to ALL Triplets
print("🔄 Adding negatives to all triplets...")

def add_negatives(triplets, neg_pool, n_negatives=1):
    for triplet in triplets:
        negs = []
        while len(negs) < n_negatives:
            neg = random.choice(neg_pool)
            if neg != triplet['positive']:
                negs.append(neg)
        triplet['negatives'] = negs
    return triplets

ms_marco_triplets = add_negatives(ms_marco_triplets, negative_pool)
jurafsky_triplets = add_negatives(jurafsky_triplets, negative_pool)
# ArXiv triplets already have negatives from previous step, but let's ensure they are consistent
# If they don't have negatives, add them (though they should already have them)
if arxiv_triplets and 'negatives' not in arxiv_triplets:
    arxiv_triplets = add_negatives(arxiv_triplets, negative_pool)

print(f"✅ Added negatives to all triplets")

# 8. Combine ALL Triplets
all_triplets = arxiv_triplets + ms_marco_triplets + jurafsky_triplets
print(f"\n📊 Total Combined Triplets: {len(all_triplets)}")
print(f"   - ArXiv: {len(arxiv_triplets)}")
print(f"   - MS MARCO: {len(ms_marco_triplets)}")
print(f"   - Jurafsky: {len(jurafsky_triplets)}")

# 9. Split into Train/Val (90/10)
train_triplets, val_triplets = train_test_split(
    all_triplets, 
    test_size=0.1, 
    random_state=42
)

print(f"\n📂 Final Split:")
print(f"   - Training Triplets: {len(train_triplets)}")
print(f"   - Validation Triplets: {len(val_triplets)}")

# 10. Save for inspection
OUT_COMBINED_TRAIN = project_root / "data" / "processed" / "combined_train_triplets.jsonl"
OUT_COMBINED_VAL   = project_root / "data" / "processed" / "combined_val_triplets.jsonl"

def write_triplets(triplets, path):
    with open(path, 'w', encoding='utf-8') as f:
        for t in triplets:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"  wrote {len(triplets):,} triplets → {path.name}")

write_triplets(train_triplets, OUT_COMBINED_TRAIN)
write_triplets(val_triplets, OUT_COMBINED_VAL)

print("✅ Data preparation complete! Ready for training.")


📚 Loading MS MARCO data directly from JSON files...
✅ Loaded 20000 queries
✅ Loaded 10000 passages
✅ Loaded 20000 relevance judgments
🔄 Generating MS MARCO triplets...
✅ Generated 20000 MS MARCO pairs
📚 Loading ArXiv triplets...
✅ Loaded 13500 ArXiv triplets
📚 Generating Jurafsky Book triplets...
✅ Generated 11607 Jurafsky triplets
🔄 Preparing negative pool...
✅ Negative pool size: 419061 passages
🔄 Adding negatives to all triplets...
✅ Added negatives to all triplets

📊 Total Combined Triplets: 45107
   - ArXiv: 13500
   - MS MARCO: 20000
   - Jurafsky: 11607

📂 Final Split:
   - Training Triplets: 40596
   - Validation Triplets: 4511
  wrote 40,596 triplets → combined_train_triplets.jsonl
  wrote 4,511 triplets → combined_val_triplets.jsonl
✅ Data preparation complete! Ready for training.


In [81]:
import random
from sklearn.model_selection import train_test_split

# 1. Combine all triplets
all_triplets = arxiv_triplets + ms_marco_triplets + jurafsky_triplets

# 2. Deduplicate based on Query Text
# We keep only the first occurrence of each unique query
seen_queries = set()
unique_triplets = []

for triplet in all_triplets:
    q = triplet['query']
    if q not in seen_queries:
        seen_queries.add(q)
        unique_triplets.append(triplet)

print(f"📉 Reduced from {len(all_triplets)} to {len(unique_triplets)} unique triplets")

# 3. Shuffle thoroughly
random.seed(42)
random.shuffle(unique_triplets)

# 4. Split
train_triplets, val_triplets = train_test_split(
    unique_triplets, 
    test_size=0.2, 
    random_state=42
)

# 5. Verify No Leakage
train_queries = set([t['query'] for t in train_triplets])
val_queries = set([t['query'] for t in val_triplets])

overlap = train_queries.intersection(val_queries)
print(f"✅ Overlap between Train and Val queries: {len(overlap)}")

if len(overlap) > 0:
    print("⚠️ WARNING: Leakage detected!")
else:
    print("✅ No leakage detected. Safe to train.")


📉 Reduced from 45107 to 25250 unique triplets
✅ Overlap between Train and Val queries: 0
✅ No leakage detected. Safe to train.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from pathlib import Path
import sys
import time

# Ensure src is in path
sys.path.insert(0, str(project_root))

# 1. Import Model and Tokenizer
from src.models.encoder import EncoderOnlyTransformer, SimpleTokenizer

# 2. Import Loss Function
from src.models.loss import InfoNCELoss

# 3. Define Configuration - Optimized for Anti-Overfitting
CONFIG = {
    'vocab_size': 20000,
    'd_model': 64,      # 🔽 Reduced from 128 to 64 (less capacity to memorize)
    'n_heads': 2,       # Kept at 4 (64/4 = 16 dims per head)
    'n_layers': 2,      # Kept at 2 (shallow network)
    'd_ff': 256,        # 🔽 Reduced from 512 to 256 (4 * d_model)
    'max_len': 128,     # Kept at 128
    'dropout': 0.5,     # 🔼 Increased from 0.3 to 0.5 (stronger regularization)
    'pad_token_id': 0
}

TRAINING_CONFIG = {
    'epochs': 1,        # 🔼 More epochs, but we'll use early stopping
    'batch_size': 256,  # 🔼 Increased from 128 to 256 (more in-batch negatives)
    'learning_rate': 1e-4,
    'temperature': 1, # 🔽 Lowered from 0.07 (focus on hard negatives)
    'weight_decay': 1e-4, # 🔼 Added for optimizer
    'gradient_accumulation_steps': 8 # Simulate batch size 256

}


# 4. Initialize Model and Tokenizer
print("🧠 Initializing Encoder Model...")
model = EncoderOnlyTransformer(CONFIG)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to_device(device)
print(f"🚀 Using device: {device}")

# 5. Prepare Data for Training
print("🔄 Fitting Tokenizer on training data...")
all_texts_for_vocab = []
for t in train_triplets:
    all_texts_for_vocab.append(t['query'])
    all_texts_for_vocab.append(t['positive'])
    # Ensure we only add strings to vocab
    for neg in t['negatives']:
        if isinstance(neg, str):
            all_texts_for_vocab.append(neg)

tokenizer = SimpleTokenizer(CONFIG['vocab_size'])
tokenizer.fit(all_texts_for_vocab)
print(f"✅ Vocabulary size: {len(tokenizer.word_to_id)}")

# 6. Create DataLoader-like Generator with Safety Checks
def generate_batches(triplets, tokenizer, batch_size, max_len):
    """Generator that yields batches of tokenized triplets"""
    random.shuffle(triplets) # Shuffle once per call
    for i in range(0, len(triplets), batch_size):
        batch_triplets = triplets[i:i+batch_size]
        if not batch_triplets:
            break
            
        queries = [t['query'] for t in batch_triplets]
        positives = [t['positive'] for t in batch_triplets]
        
        # FIX: Ensure we extract the first negative string safely
        negatives_list = []
        for t in batch_triplets:
            if t['negatives'] and len(t['negatives']) > 0:
                neg_text = t['negatives']
                if isinstance(neg_text, str):
                    negatives_list.append(neg_text)
                else:
                    # Fallback if negative is not a string
                    negatives_list.append("UNK") 
            else:
                # Fallback if no negative provided
                negatives_list.append("UNK")
        
        # Tokenize
        q_encoded = tokenizer(queries, padding=True, truncation=True, max_length=max_len, return_tensors='pt')
        p_encoded = tokenizer(positives, padding=True, truncation=True, max_length=max_len, return_tensors='pt')
        n_encoded = tokenizer(negatives_list, padding=True, truncation=True, max_length=max_len, return_tensors='pt')
        
        yield {
            'query_ids': q_encoded['input_ids'],
            'query_mask': q_encoded['attention_mask'],
            'pos_ids': p_encoded['input_ids'],
            'pos_mask': p_encoded['attention_mask'],
            'neg_ids': n_encoded['input_ids'],
            'neg_mask': n_encoded['attention_mask']
        }

# 7. Training Loop
print("🚀 Starting Training...")
print("="*50)

optimizer = optim.AdamW(model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=1e-4)
criterion = InfoNCELoss(temperature=TRAINING_CONFIG['temperature'])

model.train()
num_epochs = TRAINING_CONFIG['epochs']
batch_size = TRAINING_CONFIG['batch_size']
max_len = CONFIG['max_len']

start_time = time.time()

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    # Create a NEW generator for this epoch to ensure shuffling
    data_gen = generate_batches(train_triplets, tokenizer, batch_size, max_len)
    
    # Iterate through all batches in the generator
    for batch_idx, batch in enumerate(data_gen):
        
        # Move to device
        q_ids = batch['query_ids'].to(device)
        q_mask = batch['query_mask'].to(device)
        
        p_ids = batch['pos_ids'].to(device)
        p_mask = batch['pos_mask'].to(device)
        
        n_ids = batch['neg_ids'].to(device)
        n_mask = batch['neg_mask'].to(device)
        
        # Convert masks to boolean (True for padding) as expected by model
        q_mask_bool = (q_mask == 0)
        p_mask_bool = (p_mask == 0)
        n_mask_bool = (n_mask == 0)
        
        # Forward Pass
        anchor_emb = model(q_ids, q_mask_bool)
        pos_emb = model(p_ids, p_mask_bool)
        neg_emb = model(n_ids, n_mask_bool)
        
        # Reshape negatives for InfoNCELoss: [batch, 1, dim]
        neg_emb = neg_emb.unsqueeze(1)
        
        # Compute Loss
        loss = criterion(anchor_emb, pos_emb, neg_emb)
        
        # Backward Pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx} | Loss: {loss.item():.4f}")
    
    avg_loss = epoch_loss / num_batches if num_batches > 0 else 0
    print(f"🏁 Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")
    
    # Validation Step
    if val_triplets:
        model.eval()
        val_loss = 0.0
        val_batches = 0
        
        # Create a NEW generator for validation
        val_gen = generate_batches(val_triplets, tokenizer, batch_size, max_len)
        
        with torch.no_grad():
            for batch in val_gen:
                v_q = batch['query_ids'].to(device)
                v_p = batch['pos_ids'].to(device)
                v_n = batch['neg_ids'].to(device)
                
                v_q_bool = (batch['query_mask'] == 0).to(device)
                v_p_bool = (batch['pos_mask'] == 0).to(device)
                v_n_bool = (batch['neg_mask'] == 0).to(device)
                
                v_anchor = model(v_q, v_q_bool)
                v_pos = model(v_p, v_p_bool)
                v_neg = model(v_n, v_n_bool).unsqueeze(1)
                
                v_loss = criterion(v_anchor, v_pos, v_neg)
                val_loss += v_loss.item()
                val_batches += 1
        
        if val_batches > 0:
            print(f"📊 Validation Loss: {(val_loss/val_batches):.4f}")
            
        model.train()

total_time = time.time() - start_time
print(f"\n✅ Training Complete in {total_time/60:.1f} minutes!")

# 8. Save Model
print(f"💾 Saving model to {MODEL_SAVE_PATH}...")
model.save(str(MODEL_SAVE_PATH))
print("✅ Model Saved!")


🧠 Initializing Encoder Model...
🚀 Using device: cpu
🔄 Fitting Tokenizer on training data...
✅ Vocabulary size: 20000
🚀 Starting Training...
Epoch 1/2 | Batch 0 | Loss: 0.7085
Epoch 1/2 | Batch 50 | Loss: 0.6532
🏁 Epoch 1 Complete | Avg Loss: 0.6606
📊 Validation Loss: 0.7173
Epoch 2/2 | Batch 0 | Loss: 0.6004
Epoch 2/2 | Batch 50 | Loss: 0.5309
🏁 Epoch 2 Complete | Avg Loss: 0.5433
📊 Validation Loss: 0.1591

✅ Training Complete in 5.0 minutes!
💾 Saving model to c:\Users\salo\NLP_final_project\models\encoder_transformer.pth...
Model saved to c:\Users\salo\NLP_final_project\models\encoder_transformer.pth
✅ Model Saved!


In [87]:
import json
import sys
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# 1. Setup Path
PROJECT_ROOT = Path(r"C:\Users\salo\NLP_final_project")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.encoder import EncoderOnlyTransformer, SimpleTokenizer

# 2. Load Model & Tokenizer
MODEL_PATH = PROJECT_ROOT / "models" / "encoder_transformer.pth"
CONFIG = {
    'vocab_size': 20000,
    'd_model': 64,      # 🔽 Reduced from 128 to 64 (less capacity to memorize)
    'n_heads': 2,       # Kept at 4 (64/4 = 16 dims per head)
    'n_layers': 2,      # Kept at 2 (shallow network)
    'd_ff': 256,        # 🔽 Reduced from 512 to 256 (4 * d_model)
    'max_len': 128,     # Kept at 128
    'dropout': 0.5,     # 🔼 Increased from 0.3 to 0.5 (stronger regularization)
    'pad_token_id': 0
}

print("🧠 Loading Model...")
model = EncoderOnlyTransformer.load(str(MODEL_PATH), device=torch.device('cpu'))
model.eval()

print("🔄 Fitting Tokenizer...")
with open(PROJECT_ROOT / "data" / "processed" / "combined_train_triplets.jsonl", 'r', encoding='utf-8') as f:
    triplets = [json.loads(l) for l in f if l.strip()]
tokenizer = SimpleTokenizer(CONFIG['vocab_size'])
tokenizer.fit([t['query'] for t in triplets] + [t['positive'] for t in triplets])

# 3. Load & Encode Passages
print("📚 Loading & Encoding Passages...")
passages = []
with open(PROJECT_ROOT / "data" / "jurafsky_chunks" / "chunks.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            data = json.loads(line)
            if 'text' in data:
                passages.append(data['text'])
        except json.JSONDecodeError:
            continue # Skip malformed lines
print('yellow')
#print(passages[0])
print('aaa')

batch_size = 64
embeddings = []
for i in range(0, len(passages), batch_size):
    batch = passages[i:i+batch_size]
    emb = model.encode(batch, tokenizer)
    embeddings.append(emb)
passage_embeddings = np.vstack(embeddings)
print(f"✅ Encoded {len(passages)} passages")

# 4. Interactive Search Loop
print("\n" + "="*50)
print("🔍 INTERACTIVE NEURAL SEARCH")
print("="*50)
print("Type 'quit' to exit.\n")

for i in range(2):
    try:
        query = input("Enter query: ")
    except EOFError:
        break
        
    if query.lower() == 'quit':
        print("👋 Exiting...")
        break
        
    if not query:
        continue

    # Encode Query
    query_emb = model.encode([query], tokenizer)
    
    # Calculate Similarities
    # FIX: cosine_similarity returns (1, N). We MUST take  to get the 1D array of scores.
    similarities = cosine_similarity(query_emb, passage_embeddings)
    
    # Get Top 5 Indices
    top_indices = np.argsort(similarities)[::-1][:5]
    print('ssssss')
    print(top_indices)
    # Print Results
    print(passages[top_indices[0][0]])


print("✅ Search Complete!")


🧠 Loading Model...
Model loaded from C:\Users\salo\NLP_final_project\models\encoder_transformer.pth
🔄 Fitting Tokenizer...
📚 Loading & Encoding Passages...
yellow
aaa
✅ Encoded 1730 passages

🔍 INTERACTIVE NEURAL SEARCH
Type 'quit' to exit.

ssssss
[[1623 1559  168 ...  225 1699  520]]
tion, Duisburg, Germany, 24–26 March, pp. 72–81. Mitamura, T. and Nyberg, E. H. (1995). Controlled English fo r knowledge-based MT: Experience with the KANT system. In 6th International Conference on The- oretical and Methodological Issues in Machine Translation. Mitchell, D. C., Cuetos, F., Corley, M. M. B., and Brysbaert,M. (1995). Exposure- based models of human parsing: Evidence for the use of coarse -grained (nonlexi- cal) statistical records. Journal of Psycholinguistic Research, 24(6), 469–488. Mitchell, T. M. (1981). Generalization as search. In Webber , B. L. and Nilsson, N. J. (Eds.), Readings in Artiﬁcial Intelligence, pp. 517–542. Morgan Kaufmann, Los Altos. Mitkov, R. and Boguraev, B. (Eds.)